# VAE Training Notebook — HPC-Ready

Complete, resumable training workflow for `ConvVAE3D` / `UNetVAE3D`.  
Runs on an HPC compute node where the Jupyter kernel has direct GPU access.

**Prerequisites:**
- Clone this repo on the HPC node and install: `pip install -e .`
- Copy your processed dataset (Zarr + Parquet + splits.json) to the HPC filesystem
- Set `POREGEN_DATA_ROOT` and `POREGEN_RUN_DIR` env vars (or edit the config cell)

---
## A) HPC + Environment Check

In [ ]:
import sys, os, platform

print(f"Python   : {sys.version}")
print(f"Platform : {platform.node()} ({platform.system()} {platform.release()})")

# ── PyTorch + CUDA ──
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"cuDNN    : {torch.backends.cudnn.version()}")
print(f"CUDA available : {torch.cuda.is_available()}")

assert torch.cuda.is_available(), (
    "No CUDA device found! Make sure you're running on a compute node with GPUs. "
    "On SLURM: srun --gres=gpu:1 --pty jupyter-lab"
)

N_GPUS = torch.cuda.device_count()
print(f"GPUs ({N_GPUS}):")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {props.name}  ({props.total_mem / 1e9:.1f} GB, sm_{props.major}{props.minor})")

# ── poregen package ──
# The repo must be cloned on the HPC and installed with: pip install -e .
# This makes all poregen.* imports available to the Jupyter kernel.
try:
    import poregen
    print(f"\nporegen  : imported OK")
except ImportError:
    raise ImportError(
        "Cannot import `poregen`. On your HPC node, run:\n"
        "  cd /path/to/GenAI && pip install -e .\n"
        "Then restart the Jupyter kernel."
    )

---
## B) Configuration

All hyperparameters load from a single YAML file (`vae_default.yaml`).  
Override per-run values (paths, step count, experiment name) in the two cells below.  
Never edit the YAML for one-off experiments — use the override dict instead.


In [ ]:
from poregen.configs import load_config

cfg = load_config("../src/poregen/configs/vae_default.yaml")

# ── Per-run overrides (paths + experiment identity) ───────────────────
# Set these env vars on the HPC node, or change the fallbacks below.
DATA_ROOT = os.environ.get("POREGEN_DATA_ROOT", "../data/processed")
RUN_DIR   = os.environ.get("POREGEN_RUN_DIR",   "../runs/vae")
EXP_NAME  = "conv_baseline"  # subdirectory inside RUN_DIR

# Optional step-count override for shorter debug runs:
# cfg["training"]["total_steps"] = 5000

print(f"Config loaded — kl_max_beta={cfg['loss']['kl_max_beta']}  "
      f"z_channels={cfg['model']['z_channels']}  "
      f"total_steps={cfg['training']['total_steps']:,}")


In [ ]:
# β-schedule preview — shows the KL warm-up ramp before you start training
import matplotlib.pyplot as plt
import numpy as np
from poregen.losses.kl import beta_schedule

warmup = cfg["loss"]["kl_warmup_steps"]
max_beta = cfg["loss"]["kl_max_beta"]
total = cfg["training"]["total_steps"]
steps = np.arange(0, total, max(1, total // 2000))
betas = [beta_schedule(int(s), warmup, max_beta) for s in steps]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(steps, betas, lw=1.5, color="steelblue")
ax.axhline(max_beta, ls="--", color="gray", lw=0.8, label=f"max β={max_beta}")
ax.axvline(warmup, ls=":", color="tomato", lw=0.8, label=f"warmup={warmup:,} steps")
ax.set(xlabel="Training step", ylabel="β", title="KL β warm-up schedule")
ax.legend()
plt.tight_layout()
plt.show()


---
## C) Dataset + DataLoaders

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from poregen.dataset.loader import PatchDataset
from poregen.training import seed_everything

seed_everything(cfg["training"]["seed"])

data_root = Path(DATA_ROOT)
assert (data_root / "patch_index.parquet").exists(), (
    f"patch_index.parquet not found in {data_root}. "
    f"Copy your processed dataset to the HPC or set POREGEN_DATA_ROOT."
)

train_ds = PatchDataset(data_root / "patch_index.parquet", data_root, split="train")
val_ds   = PatchDataset(data_root / "patch_index.parquet", data_root, split="val")
test_ds  = PatchDataset(data_root / "patch_index.parquet", data_root, split="test")

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")

dl_kwargs = dict(
    batch_size=cfg["data"]["batch_size"],
    num_workers=cfg["data"]["num_workers"],
    pin_memory=cfg["data"]["pin_memory"],
    prefetch_factor=2,
    persistent_workers=cfg["data"]["num_workers"] > 0,
)

train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True, **dl_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kwargs)

# ── Verify a batch ──
sample = next(iter(train_loader))
P = cfg["model"]["patch_size"]
B = cfg["data"]["batch_size"]
print(f"xct  shape: {sample['xct'].shape}  dtype: {sample['xct'].dtype}")
print(f"mask shape: {sample['mask'].shape}  dtype: {sample['mask'].dtype}")
assert sample['xct'].shape == (B, 1, P, P, P), f"Unexpected shape: {sample['xct'].shape}"
print(f"Mean porosity: {sample['porosity'].mean():.4f}")


---
## D) Model Creation

In [ ]:
from poregen.models.vae import build_vae
from poregen.training import select_device, get_autocast_dtype, make_scaler

device = select_device()  # GPU 0 by default
autocast_dtype = get_autocast_dtype(device)
print(f"Device: {device}  |  AMP dtype: {autocast_dtype}")

mc = cfg["model"]
model = build_vae(
    mc["name"],
    z_channels=mc["z_channels"],
    base_channels=mc["base_channels"],
    n_blocks=mc["n_blocks"],
    patch_size=mc["patch_size"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {type(model).__name__}")
print(f"Parameters: {n_params:,} total  |  {n_train:,} trainable")
lat_dim = mc["patch_size"] // (2 ** mc["n_blocks"])
print(f"Latent shape: (B, {mc['z_channels']}, {lat_dim}, {lat_dim}, {lat_dim})")


---
## E) Loss + Metrics Setup

In [ ]:
import functools
from poregen.losses import compute_total_loss
from poregen.metrics import segmentation_metrics, mae, psnr
from poregen.metrics.latent import active_units

loss_fn = functools.partial(compute_total_loss, cfg=cfg)

print("Loss configuration:")
for k, v in cfg["loss"].items():
    print(f"  {k}: {v}")


---
## F) Optimizer + Scheduler

In [ ]:
tc = cfg["training"]

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=tc["lr"],
    weight_decay=1e-5,
)

scaler = make_scaler(device)

# ── LR scheduler (cosine) ──
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=tc["total_steps"], eta_min=tc["lr"] * 0.01,
)

print(f"Optimizer: AdamW (lr={tc['lr']})")
print(f"Scheduler: CosineAnnealingLR (T_max={tc['total_steps']:,})")


---
## G) Resumable Training — Checkpoint Setup

Auto-resumes from `last.ckpt` if it exists in the run directory.

In [ ]:
import subprocess
from poregen.training import load_checkpoint
from torch.utils.tensorboard import SummaryWriter

run_dir = Path(RUN_DIR) / EXP_NAME
run_dir.mkdir(parents=True, exist_ok=True)

# ── TensorBoard (real-time, background process) ────────────────────────
TB_PORT = 6007
tb_proc = subprocess.Popen(
    ["tensorboard", "--logdir", str(run_dir), "--port", str(TB_PORT), "--host", "127.0.0.1"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print(f"TensorBoard started (PID {tb_proc.pid}) → http://127.0.0.1:{TB_PORT}")
print(f"  Forward port on your laptop: ssh -N -L {TB_PORT}:127.0.0.1:{TB_PORT} <login_node>")

tb_writer = SummaryWriter(log_dir=str(run_dir))

# ── Auto-resume ──
start_step = 0
last_ckpt  = run_dir / "last.ckpt"

if last_ckpt.exists():
    start_step, meta = load_checkpoint(
        last_ckpt, model, optimizer, scaler,
        scheduler=scheduler, map_location=device, restore_rng=True,
    )
    print(f"Resumed from step {start_step:,}")
else:
    print("No checkpoint found — starting from step 0")

print(f"Run directory: {run_dir}")


---
## H) Training Loop (Step-Based Epochs)

Each epoch = `steps_per_epoch` optimizer updates.  
DataLoader cycles infinitely — no dependency on dataset size for timing.

In [ ]:
from poregen.training.engine import train_loop

tc = cfg["training"]

history = train_loop(
    model, train_loader, val_loader,
    optimizer, scaler, loss_fn,
    # ── Step counts ──
    total_steps=tc["total_steps"],
    start_step=start_step,
    # ── Logging / eval cadence ──
    log_every=tc["log_every"],
    eval_every=tc["eval_every"],
    val_batches=tc["val_batches"],
    save_every=tc["save_every"],
    image_log_every=tc["image_log_every"],
    # ── Test set evaluation ──
    test_loader=test_loader,
    test_every=tc["test_every"],
    test_batches=tc["test_batches"],
    # ── Device + run dir ──
    run_dir=run_dir,
    device=device,
    autocast_dtype=autocast_dtype,
    # ── Gradient clipping + scheduler ──
    max_grad_norm=tc["max_grad_norm"],
    scheduler=scheduler,
    # ── TensorBoard writer ──
    tb_writer=tb_writer,
)

tb_writer.close()
tb_proc.terminate()
print(f"Training complete. {len([r for r in history if r['split'] == 'train']):,} train records logged.")


---
## I) Optional: Multi-GPU (DDP)

**Recommended approach for notebooks:** convert to script and use `torchrun`.

```bash
# 1) Convert this notebook to a Python script
jupyter nbconvert --to script notebooks/train_vae.ipynb --output train_vae_script

# 2) Launch with torchrun (2 GPUs)
torchrun --nproc_per_node=2 notebooks/train_vae_script.py
```

If you want to stay in-notebook, the cell below uses `mp.spawn` to wrap the
model in DDP. **This is experimental** — `torchrun` is more reliable.

In [ ]:
# ── DDP from notebook (experimental) ──
# Only run this cell if you want multi-GPU without leaving the notebook.
# Recommended: use torchrun (see markdown above) instead.

import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler


def ddp_train_worker(rank, world_size, cfg_, run_dir_):
    """Single DDP worker — one per GPU.

    All state is reconstructed from *cfg_* inside the worker so that no
    notebook-scope globals are captured (avoids closure/pickling bugs).
    """
    import functools, os
    from pathlib import Path
    from poregen.training import seed_everything, make_scaler, get_autocast_dtype
    from poregen.models.vae import build_vae
    from poregen.dataset.loader import PatchDataset
    from poregen.losses import compute_total_loss
    from poregen.training.engine import train_loop

    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    seed_everything(cfg_["training"]["seed"] + rank)

    mc = cfg_["model"]
    model = build_vae(
        mc["name"], z_channels=mc["z_channels"],
        base_channels=mc["base_channels"], n_blocks=mc["n_blocks"],
        patch_size=mc["patch_size"],
    ).to(device)
    model = DDP(model, device_ids=[rank])

    data_root = Path(os.environ.get("POREGEN_DATA_ROOT", "../data/processed"))
    train_ds = PatchDataset(data_root / "patch_index.parquet", data_root, split="train")
    sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
    dc = cfg_["data"]
    train_loader_ = DataLoader(
        train_ds, batch_size=dc["batch_size"], sampler=sampler,
        num_workers=dc["num_workers"], pin_memory=True, drop_last=True,
    )

    tc = cfg_["training"]
    optimizer_ = torch.optim.AdamW(model.parameters(), lr=tc["lr"], weight_decay=1e-5)
    scaler_ = make_scaler(device)
    autocast_dt = get_autocast_dtype(device)
    # Each worker constructs its own loss_fn — no global capture
    loss_fn_ = functools.partial(compute_total_loss, cfg=cfg_)

    train_loop(
        model, train_loader_, None, optimizer_, scaler_, loss_fn_,
        total_steps=tc["total_steps"],
        log_every=tc["log_every"],
        save_every=tc["save_every"],
        run_dir=Path(run_dir_),
        device=device,
        autocast_dtype=autocast_dt,
        max_grad_norm=tc["max_grad_norm"],
        tb_writer=None,  # only rank 0 would log; skip for simplicity
    )

    dist.destroy_process_group()


if N_GPUS >= 2:
    print(f"Launching DDP training on {N_GPUS} GPUs...")
    mp.spawn(ddp_train_worker, args=(N_GPUS, cfg, str(run_dir)), nprocs=N_GPUS, join=True)
    print("DDP training complete.")
else:
    print(f"Only {N_GPUS} GPU(s) — use single-GPU training (section H).")


---
## J) Evaluation / Sanity Checks

### J.1) Quick Loss Plot

In [ ]:
import matplotlib.pyplot as plt

train_h = [r for r in history if r["split"] == "train"]
if train_h:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

    axes[0].plot([r["step"] for r in train_h], [r["total"] for r in train_h], lw=0.6)
    axes[0].set(xlabel="Step", ylabel="Total loss", title="Total Loss")

    axes[1].plot([r["step"] for r in train_h], [r["xct_loss"] for r in train_h], lw=0.6, label="xct")
    axes[1].plot([r["step"] for r in train_h], [r.get("mask_bce", 0) for r in train_h], lw=0.6, label="mask_bce")
    axes[1].set(xlabel="Step", title="Component Losses")
    axes[1].legend(fontsize=8)

    axes[2].plot([r["step"] for r in train_h], [r.get("kl", 0) for r in train_h], lw=0.6, color="red")
    axes[2].set(xlabel="Step", title="KL Divergence")

    plt.tight_layout()
    plt.show()
else:
    print("No training history to plot.")

### J.2) Visualize Reconstruction Slices

In [ ]:
model.eval()
with torch.no_grad():
    vis_batch = next(iter(val_loader))
    xct_in = vis_batch["xct"].to(device)
    mask_in = vis_batch["mask"].to(device)

    with torch.autocast(device_type=device.type, dtype=autocast_dtype):
        out = model(xct_in, mask_in)

    xct_pred = torch.sigmoid(out.xct_logits).float().cpu()
    mask_pred = (torch.sigmoid(out.mask_logits) > 0.5).float().cpu()

# Show first 4 samples, mid-slice
n_show = min(4, xct_pred.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(12, 3 * n_show))
if n_show == 1:
    axes = axes[None, :]

for i in range(n_show):
    mid = xct_pred.shape[2] // 2
    axes[i, 0].imshow(vis_batch["xct"][i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 0].set_title("XCT GT")
    axes[i, 1].imshow(xct_pred[i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("XCT Pred")
    axes[i, 2].imshow(vis_batch["mask"][i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("Mask GT")
    axes[i, 3].imshow(mask_pred[i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 3].set_title("Mask Pred")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Reconstruction — Mid-Slice (Z)")
plt.tight_layout()
plt.show()

### J.3) Dice Score on Validation Batches

In [ ]:
model.eval()
all_dice = []
all_iou = []
n_eval_batches = 10

with torch.no_grad():
    for i, vb in enumerate(val_loader):
        if i >= n_eval_batches:
            break
        xct_in = vb["xct"].to(device)
        mask_in = vb["mask"].to(device)
        with torch.autocast(device_type=device.type, dtype=autocast_dtype):
            out = model(xct_in, mask_in)
        seg = segmentation_metrics(out.mask_logits.float(), mask_in.float())
        all_dice.append(seg["dice_all"])
        all_iou.append(seg["iou_all"])

print(f"Dice (all, {n_eval_batches} batches): {sum(all_dice)/len(all_dice):.4f}")
print(f"IoU  (all, {n_eval_batches} batches): {sum(all_iou)/len(all_iou):.4f}")

### J.5) Per-Channel KL Bar Chart

Visualise the per-channel KL values from the last validation batch.  
Channels with KL ≤ `free_bits` (0.25) are effectively inactive and shown in orange.  
**Ablation trigger**: if fewer than 5/8 channels are active, consider dropping to `z_channels=4`.


In [ ]:
import matplotlib.pyplot as plt
from poregen.losses.kl import kl_divergence

model.eval()
with torch.no_grad():
    vb = next(iter(val_loader))
    xct_in  = vb["xct"].to(device)
    mask_in = vb["mask"].to(device)
    with torch.autocast(device_type=device.type, dtype=autocast_dtype):
        out = model(xct_in, mask_in)
    _, _, kl_per_ch = kl_divergence(
        out.mu.float(), out.logvar.float(),
        free_bits=cfg["loss"]["kl_free_bits"],
    )

free_bits = cfg["loss"]["kl_free_bits"]
kl_vals = kl_per_ch.cpu().numpy()
colors  = ["steelblue" if v > free_bits else "tomato" for v in kl_vals]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(kl_vals)), kl_vals, color=colors)
ax.axhline(free_bits, ls="--", color="gray", lw=0.8, label=f"free_bits={free_bits}")
ax.set(xlabel="Latent channel", ylabel="Mean KL", title="Per-channel KL (blue=active, red=collapsed)")
ax.legend()
n_active = sum(v > free_bits for v in kl_vals)
print(f"Active channels: {n_active}/{len(kl_vals)} "
      f"({'OK' if n_active >= 5 else '⚠ consider z_channels=4'})")
plt.tight_layout()
plt.show()


### J.4) Overfit Sanity Test (Optional)

Re-run training with `CFG["overfit_one_batch"] = True` above.  
If the model is correct, loss should drop to near-zero within 100–200 steps.